# Multimodel Incident Resolution on Arango

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mostafaibrahim17/arango-multimodel-incident-resolution/blob/main/incident_resolution.ipynb)

An incident-resolution agent on the Arango Contextual Data Platform. A fresh alert comes in; one AQL query returns the most similar past incidents (vector search), the affected-service subgraph (graph traversal), and the on-call team (key-value lookup) — document, vector, graph, and key-value in one store, one query, one round trip. A GraphRAG knowledge graph over the runbooks then grounds a cited, natural-language fix.

**Prereqs:** `pip install -r requirements.txt`, and a `.env` with your Arango host/credentials + `OPENAI_API_KEY` + `GRAPHRAG_PROJECT`/`GRAPHRAG_DB` (see `.env.example`).</cell id="870cc6ee">

In [ ]:
import json
from openai import OpenAI
from datasets import load_dataset

# Helpers live in ingest.py so the notebook and the scripts share one source of truth.
from ingest import connect, embed, EMBED_MODEL, EMBED_DIM, DATASET

db = connect()
oai = OpenAI()
topo = json.load(open("topology.json"))
print("connected:", db.name)

## 1. Schema — one store, four shapes

A document collection for incidents, a `services` + `service_depends_on` named graph for the topology, and a `teams` collection for on-call ownership. (Re-run safe: we drop and recreate.)

In [ ]:
if db.has_graph("service_topology"):
    db.delete_graph("service_topology")
for c in ["incidents", "services", "teams", "service_depends_on"]:
    if db.has_collection(c):
        db.delete_collection(c)

incidents = db.create_collection("incidents")
services = db.create_collection("services")
teams = db.create_collection("teams")
graph = db.create_graph("service_topology")
graph.create_edge_definition(
    edge_collection="service_depends_on",
    from_vertex_collections=["services"],
    to_vertex_collections=["services"],
)
print("schema ready")

## 2. Service topology + on-call teams

Hand-authored in `topology.json` — the dataset ships no CMDB, so a small curated graph keeps the traversal legible. In production these edges come from your real service map.

In [ ]:
services.insert_many([{"_key": k, **v} for k, v in topo["services"].items()])
teams.insert_many([{"_key": k, **v} for k, v in topo["teams"].items()])
db.collection("service_depends_on").insert_many(
    [{"_from": f"services/{a}", "_to": f"services/{b}"} for a, b in topo["depends_on"]]
)
print(f"{services.count()} services, {teams.count()} teams, {db.collection('service_depends_on').count()} edges")

## 3. Tickets — load, embed, index

We pull `6StringNinja/synthetic-servicenow-incidents` (500 rows, MIT), embed `short_description + description`, keep the `resolution` text, and attach each incident to a service via its category. The vector index is created **after** the load so it trains on the full set.

In [ ]:
ds = load_dataset(DATASET, split="train")
cat2svc = topo["category_to_service"]
vectors = embed(oai, [f'{r["short_description"]} {r["description"]}' for r in ds])

incidents.insert_many([
    {
        "_key": r["number"], "short_description": r["short_description"],
        "description": r["description"], "resolution": r["resolution"],
        "category": r["category"], "assignment_group": r["assignment_group"],
        "service": cat2svc.get(r["category"], "onboarding-api"), "embedding": v,
    }
    for r, v in zip(ds, vectors)
])

incidents.add_index({
    "name": "incidents_vec", "type": "vector", "fields": ["embedding"],
    "params": {"metric": "cosine", "dimension": EMBED_DIM, "nLists": 10},
})
print(f"{incidents.count()} incidents embedded + indexed")

## 4. Resolve an alert — the multimodel query

One AQL query does three moves: `APPROX_NEAR_COSINE` (vector) finds similar incidents, `OUTBOUND ... GRAPH` (graph) expands the affected-service subgraph, and `DOCUMENT` (key-value) returns the on-call team. The full query lives in `resolver.py`.

In [ ]:
from resolver import MARQUEE, resolve
print(MARQUEE)

In [ ]:
alert = json.load(open("alert.sample.json"))
print(json.dumps(resolve(alert), indent=2))

The payload is the structured answer: ranked similar incidents (with how each was resolved), the affected-service subgraph by depth, and who to page. Next we turn that into a cited, runbook-grounded fix — the `answer()` seam in `resolver.py`, backed by a GraphRAG knowledge graph.

## 5. The runbook knowledge graph (GraphRAG)

The structured payload says *what* broke and *who* owns it. To say *how to fix it* in grounded prose, we import the runbooks in `runbooks/` into a GraphRAG knowledge graph — entities, relations, communities, and chunk embeddings — queried through the Retriever. The build runs once and is skipped if the graph already exists (`graphrag_ingest.main(reset=True)` rebuilds). Services are deployed once via the platform UI; postfixes are discovered at runtime in `graphrag.py`, never hardcoded.

In [ ]:
import graphrag_ingest
graphrag_ingest.main()  # skip-if-built; pass --reset on the CLI to rebuild

from graphrag import GRAPHRAG_PROJECT, kg_db
kdb = kg_db()
for c in ["Documents", "Entities", "Relations", "Communities"]:
    print(f"{c:12}", kdb.collection(f"{GRAPHRAG_PROJECT}_{c}").count())

## 6. The cited, grounded answer

`answer()` queries the Retriever's Local Search and grounds the next step in the **exact** root-service runbook the AQL query pinpointed (primary citation), then appends GraphRAG's related blast-radius runbooks. `corroborated()` independently checks that a cited runbook references the root or an affected service — the precise AQL surface and the semantic GraphRAG surface converging on the same incident.

In [ ]:
from resolver import answer, corroborated

payload = resolve(alert)
cited = answer(payload, alert["text"])
print(cited["text"], "\n")
for k, c in cited["citations"].items():
    tag = " (primary)" if c.get("primary") else ""
    label = c.get("file_name") or c.get("citable_url", "?")
    print(f"[{k}]{tag} {label}")
print("\ncorroboration:", corroborated(payload, cited))

## 7. Results across the alert set

The same pipeline over every alert in `alerts.json`. For each, the primary citation should be that alert's own service runbook, and corroboration should be green. Across the 8 demo alerts this holds 8/8 — two independent surfaces (precise AQL + semantic GraphRAG) agreeing every time.

In [ ]:
import json
alerts = json.load(open("alerts.json"))

print(f"{'alert':9}{'service':20}{'sev':5}{'primary runbook':40}corrob")
hits = corr = 0
for a in alerts:
    p = resolve(a)
    c = answer(p, a["text"])
    prim = next((v for v in c["citations"].values() if v.get("primary")), None)
    cb = corroborated(p, c)["agree"]
    hits += bool(prim); corr += cb
    rb = prim["file_name"] if prim else "(none)"
    print(f"{a['_key']:9}{a['service']:20}{a['severity']:5}{rb:40}{'OK' if cb else 'x'}")
print(f"\nprimary-runbook hit {hits}/{len(alerts)} · corroborated {corr}/{len(alerts)}")

### Figures

The blast radius of the headline alert and the runbook knowledge graph, rendered straight from the live deployment (`python viz.py` → `assets/`).

In [ ]:
from IPython.display import Image, display
display(Image("assets/affected-subgraph.png"))
display(Image("assets/knowledge-graph.png"))

## What this shows

One Arango Contextual Data Platform deployment resolves an operational alert end to end: vector + graph + key-value in a single AQL round trip for the structured facts (similar incidents, blast radius, on-call), and a GraphRAG knowledge graph for the grounded, cited fix — two independent surfaces that corroborate. No separate vector store, graph database, or retrieval service to stitch together. Scale knobs (ticket slice, runbook count) and extension points (LangChain / LangGraph, Arango's **Ada** assistant) are documented in the README.